# Privacy and Governance Analysis

This notebook has the intention to analyse the risks regarding privacy on the start-ups credit aplications dataset, meaning that it evaluates the privacy, regulatory and governance implications of the findings indentified in the previous project analyses.  

NovaCred is a fintech company using machine learning models to support credit approval decisions. Because credit scoring directly affects individuals' access to financial services, such systems involve sensitive personal data and significant regulatory obligations.

This notebook examines whether the dataset and analytical pipeline meet key requirements related to:
- personal data protection  
- data governance  
- regulatory compliance  
- responsible AI practices


### Relationship with Previous Analyses

This notebook builds directly on the findings of Notebook 02 (Bias Analysis) and addresses the privacy and governance obligations that those findings trigger. Credit scoring is classified as a high-risk AI system under the EU AI Act (Annex III, No. 5b), which means it is subject to both GDPR and AI Act requirements simultaneously.

Examples include:
- duplicate SSNs linked to multiple individuals  
- demographic attributes used in bias analysis  
- proxy variables that may reveal protected characteristics  
- missing governance metadata such as consent timestamps or audit logs

This analysis was performed on the dataset that was already cleaned and prepared by the data team. It will be possile to focus on privacy and governance, preventing technical mistakes related to data quality.

### Sections
0. Setup & Load Data
1. Identification of Personal Data & Classification 
2. GDPR & Regulatory Obligations¶
3. Recommended Column Treatment & Pseudonymisation 
4. Governance Gaps & Recommendations
5. Summary


---

### Regulatory Framework

| Regulation | Relevance |
|---|---|
| **GDPR** (Regulation (EU) 2016/679) | Governs all processing of personal data — lawful basis, data minimisation, storage limitation, rights of data subjects |
| **EU AI Act** (Regulation (EU) 2024/1689) | Credit scoring = high-risk AI (Annex III) — requires risk management, data governance, transparency, human oversight |

## 0. Setup and Load Data

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib

DATA_PATH = Path('../data/clean_dataset_view.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Clean dataset view not found. '
        'Make sure clean_dataset_view.csv exists in the data folder.'
    )

df = pd.read_csv(DATA_PATH)

In [2]:
df.columns
columns_table = df.columns.to_frame(index=False, name='Columns')
columns_table

,Columns
0,_id
1,spending_behavior
2,processing_timestamp
3,full_name
4,email
5,ssn
6,ip_address
7,gender
8,date_of_birth
9,zip_code


## 1. Identification of Personal Data & Classification

The first step in the privacy and governance assessment is to identify which attributes in the dataset cnstitute personal data. 

Under GDPR Art. 4(1), personal data refers to information that allows the identification of an individual, either directly (e.g. name or SSN) or indirectly through the combination of multiple attributes. For that reason, it should be treated with special security and privacy measures. In this dataset, several potential fields were identified as Personally Identifiable Information (PII). 

Based on this framework, we classify dataset attributes according to their potential identifiability risk.

After analysing the columns of the dataset, it is possible to conclude that the following columns are considered personal data. We classify each field by PII type and risk level.

| Category | Definition | Examples in Dataset |
|--------|-------------|--------------------|
| **Direct Identifier** | Uniquely identifies a person on its own | full_name, ssn, email |
| **Quasi-Identifier (QI)** | Combination with other attributes may enable identification | date_of_birth, zip_code, gender |
| **Technical Identifier** | Identifies a device or interaction | ip_address |
| **Non-Personal / Derived** | Represents system outputs rather than personal identity | loan_approved, interest_rate |

Although the attributes such as `zip code` are not considered sensitive categories under GDPR Art.9, they may act as proxy variables for protected characteristics such as race or socio economic status.
When such attributes are used in automated decision-making systems,they may contribute to indirect discrimination, which was explored in the bias analysis performed in Notebook 02.

This highlights the importance of assessing both privacy risk and algorithmic fairness risks in the governance framework. 

| Column | PII Category | Risk | Justification | Findings from Bias Analysis & Data Cleaning |
|---|---|---|---|---|
| `_id` | Direct Identifier | 🔴 High | Primary key; enables re-identification when combined with other fields. | Used as linkage key in duplicate SSN deduplication (Data Quality NB). |
| `full_name` | Direct Identifier | 🔴 High | Direct identifier; unnecessary for credit modelling. | Duplicate SSNs linked to multiple names increase re-identification risk (Data Quality NB). |
| `ssn` | Direct Identifier (Sensitive) | 🔴 High | National identifier; GDPR Art. 16 rectification rights may apply to merged records. | Duplicate SSN finding indicates integrity failure and breach risk (Data Quality NB). |
| `email` | Direct Identifier | 🔴 High | Directly identifies the applicant; unnecessary for credit modelling. | — |
| `ip_address` | Technical Identifier | 🟠 High | Identifies device or network; unnecessary for credit modelling. | — |
| `date_of_birth` | Quasi-Identifier (High Risk) | 🟠 High | Full birth date exceeds data minimisation needs — only age group is required. | Four inconsistent formats caused NaT exclusions that may distort cohort-level DI ratios (Data Quality NB). Age–approval association confirmed: r = +0.12, p < 0.01 (Bias NB §3). |
| `gender` | Quasi-Identifier (Protected Attribute) | 🟠 High | Protected attribute with confirmed disparate impact (DI = 0.77, p < 0.001). Use in model training must be reviewed and documented. | Intersectional gap largest in 25–34 cohort at 23 pp (Bias NB §§2, 5). |
| `zip_code` | Quasi-Identifier (Strong Proxy) | 🟠 High | Strongest gender proxy in the dataset — a model using `zip_code` replicates the gender gap without the gender column. | corr_gender = −0.805 (Bias NB §4). |
| `annual_income` | Quasi-Identifier (Moderate Proxy) | 🟡 Medium | Legitimate credit feature; moderately correlated with age (0.394) and outcome (0.180). | `annual_salary` merged in without a documented reconciliation rule (Data Quality NB; Bias NB §4). |
| `credit_history_months` | Quasi-Identifier (Age Proxy) | 🟡 Medium | Strongest age proxy (corr = 0.649); meaningful approval predictor. Proxy risk must be documented in model governance. | Most likely driver of the 25–34 approval disadvantage (Bias NB §§3, 4, 5). |
| `savings_balance` | Quasi-Identifier (Low-Medium Proxy) | 🟡 Medium | Standard financial feature; borderline medium proxy risk due to mild age correlation. | corr_age = 0.286, corr_outcome = 0.134 (Bias NB §4). |
| `spending_category` | Quasi-Identifier (Behavioural) | 🟡 Medium | Derived from raw `spending_behavior`; strongest outcome predictor (Cramér's V = 0.198). Governance assessment required on whether applicants were informed. | Low proxy score (0.043) but moderate demographic associations — any spending skew is amplified through the model (Bias NB §4). |
| `spending_amount` | Quasi-Identifier (Behavioural) | 🟡 Medium | Also derived from `spending_behavior`; inherits the same governance concerns as `spending_category`. | Near-zero correlations with gender, age and outcome — no independent predictive value (Bias NB §4). |
| `debt_to_income` | Quasi-Identifier (Low Risk) | 🟢 Low | Standard creditworthiness indicator; low proxy correlations. | Negative value truncation in NB01 may have suppressed real signal (Data Quality NB). |
| `loan_approved` | Decision Output | 🟡 Medium | Automated decision outcome requiring auditability under GDPR Art. 22 and the EU AI Act. | All bias metrics in NB02 computed on this outcome; no audit trail field present (Bias NB §§2–5). |
| `processing_timestamp` | Metadata | 🟢 Low | Enables retention control, but the field is largely missing — storage limitation governance is unenforceable. | Largely absent — deletion policies cannot be enforced (Data Quality NB). |


## 2. GDPR & Regulatory Obligations

With the PII classification established, this section maps each field category to the specific GDPR articles it triggers. These obligations are the **regulatory basis** for every decision made in Section 3 — the column treatment that follows is a direct consequence of this mapping, not a separate exercise.

| Article | Key Principle | Evidence in Dataset | Recommended Action |
|---|---|---|---|
| Art. 6 | Lawfulness & Transparency | No `consent_timestamp` field — no evidence of explicit consent or documented lawful basis for processing sensitive attributes. | Add a `consent_timestamp` field and document the lawful basis for each category of personal data processed. |
| Art. 5(1)(b) | Purpose Limitation | `spending_category` and `spending_amount` derived from raw behavioural data; no documentation that applicants were informed of this use. | Document the purpose for each model feature; obtain explicit consent or justification for behavioural data use. |
| Art. 5(1)(c) | Data Minimisation | `gender` (DI = 0.77), `zip_code` (gender proxy, corr = −0.805), and `date_of_birth` (full date retained when age group suffices) all exceed minimisation needs. `spending_amount` carries no predictive value. | Remove `gender` and `zip_code` from model training. Generalise `date_of_birth` to age group. Review necessity of `spending_amount`. |
| Art. 5(1)(d) | Accuracy | `age` derived from `date_of_birth` becomes stale without dynamic recalculation. `annual_salary` merged into `annual_income` in NB01 without a documented reconciliation rule. | Recalculate derived fields dynamically; document the reconciliation rule for `annual_income`. |
| Art. 5(1)(e) | Storage Limitation | `processing_timestamp` is largely absent (NB01) — the organisation cannot demonstrate records are not retained beyond their intended purpose. | Define a retention policy; restore timestamp coverage; implement automated deletion or anonymisation schedules. |
| Art. 32 | Integrity & Confidentiality | Dataset contains `full_name`, `ssn`, `email`, `ip_address`, and sensitive financial data. No evidence of encryption, access controls, or role-based access policies. | Implement encryption at rest and in transit, role-based access control, and audit logs for all data access. |
| Art. 22 | Human Oversight — Automated Decisions | Loan approval determined through automated scoring. No human review mechanism, no override process, and no audit trail field — three compounding gaps for a high-risk AI system under the EU AI Act. | Introduce human-in-the-loop review for borderline decisions; provide explainability mechanisms; maintain a decision audit trail (EU AI Act Art. 12). |


## 3. Recommended Column Treatment & Pseudonymisation

Based on the GDPR obligations identified in Section 2 and the bias findings from Notebook 02, each column requires one of four treatments before any model training or further analytical use.

### 3.1 Delete — Direct Identifiers (Art. 5(1)(c) + Art. 32)

These fields have no modelling value and represent the highest re-identification risk. They must be removed from the analytical dataset. For record linkage purposes only, `ssn` and `_id` may be replaced with a pseudonymous token.

| Column | Reason |
|---|---|
| `full_name` | Direct identifier; duplicate SSN–name mismatches increase breach risk. |
| `email` | Direct identifier; entirely unnecessary for credit scoring. |
| `ip_address` | Technical identifier; unnecessary for credit modelling. |
| `ssn` | National identifier; duplicate values indicate integrity risk — pseudonymise for linkage only. |

---

### 3.2 Exclude from Model Training — Protected Attributes & Strong Proxies (Art. 5(1)(c) + Art. 22)

These fields must **not be used as model features** during training or inference. They may be retained in a separate, access-controlled governance dataset for bias monitoring and fairness auditing only.

| Column | GDPR / Bias Justification |
|---|---|
| `gender` | Protected attribute. DI = 0.77 confirmed (p < 0.001). Direct use constitutes unlawful discrimination without strict justification. |
| `zip_code` | Strong gender proxy (corr = −0.805). A model trained on `zip_code` replicates the gender gap without the gender column. Must be removed or generalised to a broader region. |
| `date_of_birth` | Full birth date is PII exceeding what is needed. Generalise to age group (bins: 18–24, 25–34, 35–44, 45–54, 55–64, 65+) and drop the raw date. |

#### Note on `age` — justifiable but requires scrutiny

`age_group` (generalised from `date_of_birth`) occupies a distinct position compared to `gender` and `zip_code`:

- The bias analysis confirms a small but statistically significant positive association with approval (r = +0.12, p < 0.01), with approved applicants averaging 2.7 years older than rejected ones.
- This is **mechanically explainable**: credit history length accumulates with age, and `credit_history_months` is the strongest age proxy in the dataset (corr = 0.649). The 25–34 disadvantage is most likely driven by shorter credit histories rather than age-based prejudice per se.
- Age therefore has a **legitimate, non-discriminatory justification** as an indirect credit feature — unlike gender, which has no legitimate predictive basis in the dataset.
- However, the intersectional 25–34 female gap (23 pp, χ² p = 0.008) means any model using age must be actively monitored for intersectional effects.

---

### 3.3 Transform Before Use — Quasi-Identifiers (Art. 5(1)(c))

| Column | Recommended Treatment |
|---|---|
| `date_of_birth` | Generalise to `age_group`; drop raw date after derivation |
| `zip_code` | Generalise to region (e.g. NUTS-2) or drop entirely |
| `annual_income` | Consider bucketing into income brackets to reduce granularity |
| `spending_category` | Retain for modelling but document purpose; confirm applicants were informed |
| `spending_amount` | Governance review required; no independent predictive value found in NB02 |

---

### 3.4 Privacy-Preserving Techniques

| Technique | Description | Applicable Fields |
|---|---|---|
| **Pseudonymisation / Hashing** | Replace direct identifiers with a deterministic hash (HMAC-SHA256 + salt). | `full_name`, `email`, `ssn`, `ip_address`, `_id` |
| **Tokenisation** | Replace with a random opaque token in a separate lookup table. Stronger than hashing. | `ssn` |
| **Generalisation** | Replace precise values with broader categories. | `date_of_birth` → age group, `annual_income` → bracket, `zip_code` → region |
| **Suppression / Deletion** | Remove the field entirely when there is no legitimate analytical need. | `email`, `ip_address` |
| **Differential Privacy** | Add calibrated noise to aggregate statistics or model outputs. | `annual_income`, `savings_balance` in published statistics |
| **Data Masking** | Obscure parts of a value in display contexts (e.g. last 4 digits of SSN only). | `ssn` in operational interfaces |
| **Governance Control** | Retain under access-controlled conditions for fairness auditing — excluded from training. | `gender`, `zip_code` |

---

### 3.5 Pseudonymisation Demonstration

One of the main techniques recommended by the GDPR for the protection of personal data is pseudonymization, which consists of replacing direct identifiers with artificial representations that do not allow a person to be identified without access to additional information. In the context of this credit application dataset, this is applied to attributes such as name, email address, SSN, and IP address: fields that allow for the immediate identification of an individual and represent a high risk in case of unauthorised access. By replacing these fields, the exposure of sensitive data is reduced without compromising the analytical capacity of the dataset. 

However, pseudonymized data continues to be considered personal data under GDPR, since re-identification may still be possible through combination with other variables. Pseudonymization should therefore be seen as a risk mitigation measure and not a complete anonymization solution — a distinction recognised in GDPR Recital 26.

In [6]:
def pseudonymize(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode()).hexdigest()

pii_columns = ["full_name", "email", "ssn", "ip_address"]

for col in pii_columns:
    df[col + "_pseudo"] = df[col].apply(pseudonymize)

df = df.drop(columns=pii_columns)
df.head()

,_id,spending_behavior,processing_timestamp,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,notes,full_name_pseudo,email_pseudo,ssn_pseudo,ip_address_pseudo
0,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,female,1986-05-27,90230,102000.0,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR,574bb517d8a0854fc6906b690627290925f1e8d5cd66f9...,137c1b4a89343995539bffbc9ce6edaa65bcb498fc0434...,d49cad59b567ffb59422b169add8cf587764aabe1b91f6...,2ac8bc04be2fbb6540f7d976fda8091de37136368fa662...
1,app_002,"[{'category': 'Education', 'amount': 533}]",NaN,male,1999-08-01,10020,41000.0,5,0.36,18200,False,algorithm_risk_score,NaN,NaN,NaN,NaN,a6df505311f83600f5db73028630064a1243ff7f5874dd...,ad586ff31ede0ddcf626645f064f888e0f8e7beb3f81e2...,0e1892639ce70228710f0735fa85377403b32b612f398c...,a96122c4b0370a28986788dc0eb2030aa9c254a19736b0...
2,app_003,"[{'category': 'Healthcare', 'amount': 450}]",NaN,female,1982-08-24,90213,65000.0,74,0.43,7090,True,NaN,NaN,3.4,76000.0,NaN,d2802474636ba97c1cd42a7dde4faba876eade9b812ff6...,f110fede2c1f19f308c6cbf664f079f53438e00dfe44d6...,5e93b5041cec1a785d060c243cd2a8f8b80493574aebdd...,a6be52c95db995dc90b7718c92acd95a5a6aa1bb852f87...
3,app_004,"[{'category': 'Transportation', 'amount': 329}...",NaN,female,02/28/1995,90217,69000.0,9,0.41,10327,False,high_dti_ratio,NaN,NaN,NaN,NaN,1d462c9ea4cb62c4b2604e5b5d3ff78b403d5cc29b96cf...,a094d9056654b513ce0e2f5c254f33470604a4549c72d3...,3ef30b5e161efd64477f8094eafe1546f79df340fa9ad6...,e640905a96e360ac1b7a7535a37024dc1875e7d06aa417...
4,app_005,"[{'category': 'Insurance', 'amount': 585}]",NaN,female,1960/06/19,90296,39000.0,76,0.06,15011,False,algorithm_risk_score,NaN,NaN,NaN,NaN,14576b60592c0da19715475876d40dcaf0625b9a78b7df...,1a407bcae335e67007abb52c9b1766b7b9cd78403829ff...,8c2a4e7afef60f4e60cccf69d18a73b7ccb2e329904802...,416311facac9e50393e42001611b32a0d7b8af4c964fd6...


## 4. Governance Gaps & Recommendations

Sections 1–3 addressed field-level decisions. This section consolidates the structural governance gaps that exist at the pipeline and process level — issues that cannot be resolved by transforming individual columns, but require organisational action. Each gap is mapped to its evidence, the regulatory article it violates, and the recommended remediation.

| Governance Gap | Evidence / Source | Regulatory Reference | Recommended Action |
|---|---|---|---|
| No consent_timestamp or documented lawful basis | No consent_timestamp column in any notebook | GDPR Art. 6 — Lawful basis; Art. 7 — Consent | Add consent_timestamp; document lawful basis per processing activity before re-use for model training. |
| Duplicate SSNs linked to multiple names | Duplicate SSNs with multiple names found in NB01 | GDPR Art. 16 — Rectification; Art. 32 — Security | Investigate root cause; review rectification rights for affected individuals; add SSN uniqueness validation. |
| `annual_income` provenance not recorded (merged from `annual_salary`) | Merged without reconciliation rule (Data Quality NB) | GDPR Art. 5(1)(b) — Purpose limitation; Art. 5(1)(d) — Accuracy | Document reconciliation rule; record data provenance in pipeline metadata. |
| `processing_timestamp` largely absent — storage limitation unenforceable | Field largely NaN — deletion schedules cannot be enforced (Data Quality NB) | GDPR Art. 5(1)(e) — Storage limitation | Restore or re-collect the field; define and automate a retention schedule. |
| `spending_category` / `spending_amount` — behavioural data without documented purpose | `spending_behavior` parsed into two fields; `spending_category` is strongest outcome predictor (Cramér's V = 0.198); `spending_amount` has zero predictive value (Bias NB §4) | GDPR Art. 5(1)(b) — Purpose limitation; Art. 5(1)(c); Art. 22 profiling | Document purpose and confirm applicants were informed. Remove `spending_amount`. Monitor `spending_category` for intersectional effects. |
| No audit trail field — EU AI Act Art. 12 non-compliance | No `decision_id`, model version, or decision log in dataset | EU AI Act Art. 12 — Record-keeping; Art. 13 — Transparency | Introduce `decision_id`, `model_version`, `decision_timestamp`; implement append-only decision logs. |
| Intersectional gap (25–34 female cohort, 23 pp) without human oversight mechanism | χ² p = 0.008, Cramér's V = 0.22 (Bias NB §5); no human review mechanism identified | GDPR Art. 22; EU AI Act Art. 14 — Human oversight | Implement human review workflow for borderline decisions, particularly in the 25–34 cohort. |
| No explainability or appeal mechanism for automated decisions | `loan_approved` has no explanation, confidence score, or appeal reference | GDPR Art. 22(3) — Right to explanation; EU AI Act Art. 13 | Implement model explainability (e.g. SHAP or LIME); provide plain-language explanations and an appeal mechanism. |


These recommendations aim to strengthen data governance practices, reduce privacy risks, and support the responsible use of personal data in automated credit decision systems, in line with both GDPR requirements and emerging AI governance frameworks such as the EU AI Act.

## 5. Summary

The governance gaps identified above are concentrated in three structural areas:

1. **Data lifecycle controls** — no consent mechanism, no retention timestamps, no deletion policy, and largely absent `processing_timestamp`. These collectively prevent NovaCred from demonstrating GDPR Art. 5(1)(e) compliance.

2. **Model training pipeline** — `gender` and `zip_code` must be removed from training features. `date_of_birth` must be generalised.

3. **Auditability and human oversight** — there is the need of audit trail field, explainability mechanism, and human review process as the credit scoring is a system classified as high-risk under Annex III. The intersectional finding in the 25–34 cohort (23 pp gender gap, statistically confirmed) makes this particularly urgent.
